# pars 

In [ ]:
import importlib
from pathlib import Path
import tmspath_utils_rest as tmsr
importlib.reload(tmsr)
date,start_time=tmsr.import_modules()

In [ ]:
json_data={
    # General
    "date":date,
    "start_time":start_time,
    "analysis_id":None,
    "showPlotsEnd":False,
    "eeg_type":"rest",

    # Directory and subject
    "mainDir":r"../data/PP064/",
    "sourceData":"MAYER",
    "dataType":"ASCII",
    "subject":"PP064REST",
    "subject_id":"PP064",

    # Filtering
    "do_filter_and_plot_raw":True,
    "r_sfreq":1024,
    "l_freq":0.1,
    "h_freq":45.0,
    "broad_band_h_freq":250.0,
    "powerline_freq":50.0,

    # Cleaning
    "do_clean_trials_channels":True,
    "do_chan_trials_selection_automatic":True,
    "rest_auto_window_sec":1.0,
    "rest_auto_channel_z_threshold":3.5,
    "rest_auto_segment_z_threshold":3.5,
    "rest_auto_protect_seed_channels":True,
    "bad_trials":[],
    "bad_channels":[],

    # REST detrending
    "do_rest_detrend": False,
    "rest_detrend_order":1,

    # Continuous recording rejection
    "rest_bad_segment_mode":"free_annotations",
    "rest_annotation_h_freq_vis":60.0,
    "rest_gui_scaling":50e-6,
    "rest_crop_edges":False,
    "rest_preselect_edges_as_bad":True,
    "rest_crop_start_sec":30.0,
    "rest_crop_end_sec":30.0,

    # Synthetic REST data
    "use_synthetic_rest_signal":False,
    "synthetic_rest_base_freq":10.0,
    "synthetic_rest_jitter_hz":5.0,
    "synthetic_rest_amplitude_uv":20.0,
    "synthetic_rest_noise_uv":5.0,
    "synthetic_rest_add_aperiodic":True,
    "synthetic_rest_aperiodic_exponent":1.5,
    "synthetic_rest_aperiodic_amplitude_uv":8.0,
    "synthetic_rest_aperiodic_fmin":0.5,
    "synthetic_rest_seed":42,
    "synthetic_rest_random_phase":True,
    "synthetic_rest_add_drift":True,
    "synthetic_rest_drift_type":"quadratic",
    "synthetic_rest_drift_amplitude_uv":100.0,

    # ICA
    "do_ica":True,
    "do_ica_automaticRej":True,
    "do_ica_manualCheck":False,
    "do_label_prob_threshold":0.80,
    "save_ica_component_batches":True,
    "save_ica_component_properties":True,
    "save_ica_kept_component_properties":True,
    "save_postICA_preview_plots":True,

    # TEP files used for REST trigger alignment
    "tep_sx_file":"PP064EMISFEROSX.EDF",
    "tep_dx_file":"PP064EMISFERODX.EDF",
    "rest_tep_epoching":True,
    "rest_tep_reject_by_annotation":True,
    "rest_tep_resample_to_tep": False,

    # Epoching shared by TEP and REST PCIst
    "epochs_timewindow_min":-0.400, # devono essere uguali a TEP -0.400
    "epochs_timewindow_max":0.400, # devono essere uguali a TEP +0.400

    # MNE epoch baseline correction
    "baseline_cor_tmin":-0.150, # devono essere uguali a TEP -0.150
    "baseline_cor_tmax":-0.000, # devono essere uguali a TEP +0.000

    # REST spectral features
    "rest_features_fmin":1.0,
    "rest_features_fmax":40.0,
    "rest_features_relative":True,
    "rest_features_remove_aperiodic":True,
    "rest_features_aperiodic_fmin":1.0,
    "rest_features_aperiodic_fmax":40.0,
    "rest_features_aperiodic_mode":"fixed",
    "rest_features_n_per_seg":2000,
    "rest_features_n_overlap":0,
    "rest_features_n_fft":2048,

    'do_standard_features': True,

    # PCIst shared by TEP and REST
    "do_pcist":True,
    "pcist_baseline_window_ms":(-150.0,-10.0),
    "pcist_response_window_ms":(10.0,300.0),
    "pcist_k":1.2,
    "pcist_min_snr":1.1,
    "pcist_max_var":99.0,
    "pcist_embed":False,
    "pcist_n_steps":100,
    "pcist_baseline_corr":False,
    "pcist_plot_max_components":6,
    "pcist_safe_margin_ms":2.0,
    # PCIst baseline sensitivity
    "pcist_baseline_sweep": False, # interessante per confronto rispetto al variare della lunghezza della baseline
    "pcist_baseline_sweep_start_ms":-300.0,
    "pcist_baseline_sweep_end_ms":-10.0,
    "pcist_baseline_sweep_step_ms":10.0,
    "pcist_baseline_sweep_min_duration_ms":50.0,
    # REST null PCIst
    "rest_compute_null_pcist": True, 
    "rest_compute_random_pcist":False, # per ora False, esegue la distribuzione nulla con random triggers sia per SX che DX - molto tempo
    "rest_random_pcist_replicates":1*25,
    "rest_random_pcist_seed":42
}


mainDir=Path(json_data["mainDir"]).expanduser()
subject=json_data["subject"]

fileName=str(mainDir/subject)
savePath=str(mainDir/subject)
experiment_dir_=str(mainDir/subject)

json_data["analysis_id"]=f"{date}_{subject}"

print("fileName:",fileName)
print("analysis_id:",json_data["analysis_id"])

# some refs
- https://www.nature.com/articles/s41597-023-02525-0
- https://www.jove.com/t/68350/pipemat-rs-development-validation-standardized-matlab-pipeline-for
- https://www.frontiersin.org/journals/neuroscience/articles/10.3389/fnins.2018.00097/full
- https://www.frontiersin.org/journals/neuroscience/articles/10.3389/fnins.2018.00513/full

# Resting-State EEG Processing Pipeline

### Loading & Filtering

* **Load continuous resting-state EEG**
* **Apply channel montage**
* **Select EEG channels**
* **Broad-band filter** (0.1–250 Hz)
* **Notch filter** at 50 Hz and harmonics

---

### Artifact Detection

* **Mark initial and final recording intervals as BAD**, or crop them
* **Automatically detect bad channels**
* **Automatically detect bad temporal segments**
* **Manually review channels and annotations**
* **Store bad channels** in `raw.info["bads"]`

Bad channels are **not interpolated** at this stage.

---

### Continuous Detrending

* *Optional:* **Polynomial detrending / DC correction**
* Exclude BAD-annotated samples from the fit
* Preserve all annotations and bad-channel labels

---

### Pre-ICA Spectral Analysis

* **Compute Welch PSD**
* **Extract frequency-band power**

  * delta
  * theta
  * alpha
  * beta
  * gamma
* *Optional:* **FOOOF aperiodic fitting and correction**

---

### ICA

* **Downsample a copy for ICA**
* **Apply average reference**
* **Fit ICA on good EEG channels only**
* Exclude BAD-annotated temporal intervals from ICA fitting
* **Automatic and manual component selection**

  * ICLabel
  * topographies
  * component PSDs
* **Apply ICA correction to the continuous signal**

---

### Finalization

* **Apply final analysis-band filter** (`l_freq`–`h_freq`)
* **Resample to the original sampling frequency**
* **Interpolate bad channels**
* **Apply final average reference**
* **Compute post-ICA PSD and band power**
* **Compare pre-ICA and post-ICA spectral measures**
* **Save final Raw, ICA model, annotations, tables and plots**

### Features Extraction

* **TEP-trigger-matched REST PCIst**, with optional random-trigger null distribution
* **Welch PSD**
* **Absolute and relative band power**
* **FOOOF aperiodic parameters**
* **Aperiodic-corrected band power**
* **Pre-ICA versus post-ICA spectral changes**



# init

In [ ]:
json_data,experiment_dir,sub=tmsr.directorySetup(
    json_data
)

# load_and_prepare_raw_data

In [ ]:
raw,events,json_data=tmsr.load_and_prepare_raw_data(
    json_data=json_data,
    fileName=fileName,
    experiment_dir=experiment_dir,
    sub=sub
)

# DESCRIZIONE
# Caricamento e preparazione del segnale REST
# Carica il segnale EEG continuo dal formato configurato,
# assegna i tipi di canale, applica o verifica il montaggio
# e recupera gli eventuali eventi disponibili.
# Questa fase conserva il segnale continuo, necessario per
# l'analisi resting-state, e aggiorna i metadati della
# registrazione nel JSON.

# computeBasicSteps

In [ ]:
raw_clean,json_data=tmsr.computeBasicSteps(
    raw=raw,
    events=events,
    json_data=json_data,
    experiment_dir=experiment_dir,
    sub=sub,
    computeFOOOF=False
)

# Preprocessing di base
# Esegue le operazioni preliminari sul segnale REST:
# - rimozione delle porzioni iniziali e finali non utilizzabili
# - filtraggio broad-band
# - notch filter alla frequenza di rete e alle armoniche
# - rilevamento o marcatura dei canali rumorosi
# - preparazione del segnale destinato all'ICA
# - preparazione del segnale per le analisi spettrali
# computeFOOOF=False evita in questa fase un'estrazione
# spettrale ridondante.


# ICAprocessing

In [ ]:
(
    raw_after_optional_ica,
    ica_model,
    json_data
)=tmsr.run_optional_rest_ica(
    raw_clean=raw_clean,
    json_data=json_data,
    experiment_dir=experiment_dir,
    sub=sub
)

# DESCRIZIONE
# ICA opzionale
# Esegue l'ICA solo quando do_ica=True.
# L'ICA viene stimata sul segnale continuo usando i soli
# canali EEG validi. I canali marcati come bad sono esclusi
# dal fit ma restano presenti nell'oggetto.
# Le componenti artefattuali possono essere selezionate
# automaticamente e/o manualmente.
# Output:
# - raw_after_optional_ica: segnale corretto oppure copia
#   del segnale non corretto
# - ica_model: modello ICA, o None se l'ICA è disattivata
# - json_data: parametri e risultati ICA aggiornati

# finalizing steps

In [ ]:
(
    rest_final,
    json_data
)=tmsr.finalize_rest_raw(
    raw_input=raw_after_optional_ica,
    json_data=json_data,
    experiment_dir=experiment_dir,
    sub=sub,
    compute_psd_plot=True
)

json_data["rest_compute_band_comparison"]=json_data["ICA_applied"]

# DESCRIZIONE
# Finalizzazione del segnale REST
# Applica le trasformazioni finali al segnale:
# - filtro finale
# - ricampionamento
# - interpolazione dei canali bad
# - riferimento medio EEG
# - salvataggio dell'oggetto finale
# - eventuale PSD finale di controllo
# Il risultato rest_final è il segnale utilizzato per tutte
# le feature resting-state successive.

# feature extraction


In [ ]:
if json_data["do_standard_features"]:
    rest_results,json_data=tmsr.extractRestFeatures(
        postICA_rest=rest_final,
        preICA_rest=raw_clean if json_data["ICA_applied"] else None,
        json_data=json_data,
        experiment_dir=experiment_dir,
        sub=sub,
        pcist_module_name="tmspath_utils",
        save=True
    )

# DESCRIZIONE
# Estrazione delle feature REST
# Calcola le feature abilitate nella configurazione.
# Le analisi possono includere:
# - PSD per canale
# - potenza assoluta e relativa nelle bande
# - correzione del background aperiodico
# - offset ed exponent aperiodici
# - confronti pre/post ICA
# - epoching REST sugli orari dei trigger TEP SX e DX
# - PCIst null/control per SX e DX
# - eventuali misure aggiuntive definite nella pipeline
# I risultati vengono salvati nelle sottocartelle di
# 5.Extra/FE e registrati nel JSON globale.

# save and loading

In [ ]:
json_data=tmsr.saveLoadTestFinal(
    final_data=rest_final,
    json_data=json_data,
    experiment_dir=experiment_dir,
    sub=sub
)

# DESCRIZIONE
# Controllo finale, salvataggio e ricaricamento
# Esegue i controlli conclusivi della pipeline:
# - genera il summary finale REST
# - salva i percorsi dei risultati nel JSON
# - verifica e registra gli oggetti finali
# - aggiorna hash e metadati dei file
# - salva il JSON globale dell'analisi
# Questa cella deve essere eseguita dopo la finalizzazione
# e l'estrazione delle feature.